<a href="https://colab.research.google.com/github/darsanipriya-13/World-energy-consumption/blob/main/Copy_of_World_energy_consumption.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.colab import files

uploaded = files.upload()

file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print(df.head())

In [ ]:
# Remove empty rows and clean country names
df = df.dropna(subset=["Country"])
df["Country"] = df["Country"].astype(str).str.strip()

# Select India
country = "India"
row = df[df["Country"] == country]

if row.empty:
    raise ValueError("Selected country not found!")

# Convert year columns into numerical values
years = [c for c in df.columns if str(c).isdigit()]

energy = pd.to_numeric(
    row.iloc[0][years], errors="coerce"
)

# Convert wide format to chronological format
data = pd.DataFrame({
    "Year": years,
    "Energy": energy.values
})

data["Year"] = data["Year"].astype(int)
data = data.sort_values("Year").reset_index(drop=True)

# Handle missing values
data["Energy"] = data["Energy"].interpolate(
    limit_direction="both"
)

data = data.dropna(subset=["Energy"])

print(data.head())
print(data.tail())
print("Number of yearly records:", len(data))
print("Missing values:", data["Energy"].isna().sum())

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(data["Year"], data["Energy"], marker="o")

plt.title("Energy Consumption in India")
plt.xlabel("Year")
plt.ylabel("Energy Consumption")
plt.grid(True)
plt.show()

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Use the first 80% of yearly observations for training
split_index = int(len(data) * 0.8)

scaler = MinMaxScaler(feature_range=(0, 1))

# Fit only on training-period data to prevent leakage
scaler.fit(data[["Energy"]].iloc[:split_index])

scaled_energy = scaler.transform(data[["Energy"]])

print("Original values:")
print(data["Energy"].head())

print("Normalized values:")
print(scaled_energy[:5])

In [ ]:
X = []
y = []
target_years = []

for i in range(5, len(scaled_energy)):
    X.append(scaled_energy[i-5:i])
    y.append(scaled_energy[i])
    target_years.append(data["Year"].iloc[i])

X = np.array(X)
y = np.array(y)
target_years = np.array(target_years)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("First 5-year input:", X[0].flatten())
print("First target year:", target_years[0])

In [ ]:
train_mask = target_years < data["Year"].iloc[split_index]

X_train = X[train_mask]
y_train = y[train_mask]

X_test = X[~train_mask]
y_test = y[~train_mask]

years_test = target_years[~train_mask]

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Test years:", years_test)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input

model = Sequential([
    Input(shape=(5, 1)),
    LSTM(32, activation="tanh"),
    Dense(1)
])

model.compile(
    optimizer="adam",
    loss="mean_squared_error"
)

model.summary()

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=8,
    verbose=1
)

In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error

# Predict normalized energy values
pred_scaled = model.predict(X_test)

# Convert predictions and targets to original scale
y_actual = scaler.inverse_transform(
    y_test.reshape(-1, 1)
).flatten()

y_predicted = scaler.inverse_transform(
    pred_scaled
).flatten()

# Calculate evaluation metrics
mae = mean_absolute_error(y_actual, y_predicted)

rmse = np.sqrt(
    mean_squared_error(y_actual, y_predicted)
)

print("MAE:", round(mae, 4))
print("RMSE:", round(rmse, 4))

In [ ]:
results = pd.DataFrame({
    "Year": years_test,
    "Actual": y_actual,
    "Predicted": y_predicted
})

print(results.round(3))

plt.figure(figsize=(10, 5))
plt.plot(results["Year"], results["Actual"],
         marker="o", label="Actual")
plt.plot(results["Year"], results["Predicted"],
         marker="x", label="Predicted")

plt.title("Actual vs Predicted Energy Consumption")
plt.xlabel("Year")
plt.ylabel("Energy Consumption")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
last_five_years = scaled_energy[-5:].reshape(1, 5, 1)

next_scaled = model.predict(last_five_years)

next_energy = scaler.inverse_transform(
    next_scaled
)

print("Predicted energy consumption for 2024:",
      round(next_energy[0][0], 3))